# A model that learns here

Define a neural network in Python, compute its gradients with **native MLX on this device’s GPU**, and save the learned parameters. This uses Vault’s explicit `vaultlab.metal` API.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vaultlab import metal as mx

print(mx.info())
rng = np.random.default_rng(7)
x = np.linspace(-2, 2, 64, dtype=np.float32).reshape(-1, 1)
y = np.sin(2 * x)
parameters = [rng.normal(0, .5, (1, 16)).astype("float32"),
              np.zeros((16,), dtype="float32"),
              rng.normal(0, .2, (16, 1)).astype("float32"),
              np.zeros((1,), dtype="float32")]

In [ ]:
def predict(p, inputs):
    w1, b1, w2, b2 = p
    return mx.tanh(mx.array(inputs) @ w1 + b1) @ w2 + b2

def loss(p, inputs, targets):
    error = predict(p, inputs) - mx.array(targets)
    return mx.mean(error * error)

loss_and_grad = mx.value_and_grad(loss)
history = []
for step in range(120):
    value, gradients = loss_and_grad(parameters, x, y)
    parameters = [p - 0.08 * g for p, g in zip(parameters, gradients)]
    history.append(value)
print(f"Loss: {history[0]:.5f} → {history[-1]:.5f}")
assert history[-1] < history[0] * .65, "Investigate the training loop"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(history, color="#527c72", lw=2)
axes[0].set(xlabel="Training step", ylabel="Mean squared error")
prediction = predict([mx.array(p) for p in parameters], x).numpy()
axes[1].plot(x[:, 0], y[:, 0], color="#344f62", label="Target")
axes[1].plot(x[:, 0], prediction[:, 0], color="#b38c5b", label="Learned", linestyle="--")
axes[1].legend(frameon=False)
for ax in axes: ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
np.savez("model-checkpoint.npz", **{f"p{i}":p for i,p in enumerate(parameters)})
with np.load("model-checkpoint.npz") as checkpoint:
    restored = [checkpoint[f"p{i}"].copy() for i in range(len(parameters))]
for a, b in zip(parameters, restored):
    np.testing.assert_array_equal(a, b)
print("Checkpoint saved, reloaded, and verified.")

## Make it your experiment

Change the hidden width, activation, target function, or learning rate. Compare against the original and explain the difference. This small controlled fixture demonstrates the local training path; it is not evidence about minds or brains.